# Is this email critical for our project?


A customer writes to us. The agent has to find the project the email belongs to, compare it with the specification, offer and contract, and apply the business rules of the ontology (email criticality, escalation, change requests).

**Prerequisites:** `./deploy.sh` was run, `az login` is active and the Fabric capacity is running.

## 1. Connect to the demo


In [1]:
# All cells share one Bash process, so variables and functions carry over.
RG=rg-projplan-demo
AGENT=project-planning-agent
output() { az deployment group show -g "$RG" -n main --query "properties.outputs.$1.value" -o tsv; }
PROJECT_ENDPOINT=$(output foundryProjectEndpoint)
CAPACITY=$(output fabricCapacityName)
echo "Project endpoint: $PROJECT_ENDPOINT"
echo "Agent:            $AGENT"
echo "Capacity state:   $(az resource show -g "$RG" -n "$CAPACITY" --resource-type Microsoft.Fabric/capacities --query properties.state -o tsv)"

Project endpoint: https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning


Agent:            project-planning-agent


Capacity state:   Active


## 2. Helper functions
`ask` sends a question to the agent through the Foundry Responses API and prints the answer. `triage` wraps an incoming email into a question that asks for project, verdict, the rule and clause behind it, and next steps with owner and deadline.


In [2]:
ask() {
  RESPONSE=$(curl -s "$PROJECT_ENDPOINT/openai/v1/responses" \
    -H "Authorization: Bearer $(az account get-access-token --scope https://ai.azure.com/.default --query accessToken -o tsv)" \
    -H "Content-Type: application/json" \
    -d "$(jq -n --arg agent "$AGENT" --arg q "$1" '{agent_reference: {type: "agent_reference", name: $agent}, input: $q}')")
  echo "$RESPONSE" | jq -r '.error.message // (.output[] | select(.type == "message") | .content[].text)'
}

triage() {
  ask "We received the email below. Answer in this structure:
1. Project and project manager.
2. Verdict: critical, watch or not critical, according to the business rules in the ontology.
3. Why: name the rule that applies and quote the clause of the specification, offer or contract that it touches.
4. Next steps: a numbered list, each step with the responsible person and the deadline.

From: $1
Subject: $2
Body: $3"
}


## 3. Email 1: a customer wants an earlier completion
The email does not name the project. The agent has to identify it from the sender.


In [3]:
FROM="client@nordbau.example"
SUBJECT="Earlier move-in"
BODY="Our tenant wants to move in on 2027-11-01 instead of January. Please bring the completion date forward by six weeks."
echo "From:    $FROM"
echo "Subject: $SUBJECT"
echo
echo "$BODY"


From:    client@nordbau.example


Subject: Earlier move-in


Our tenant wants to move in on 2027-11-01 instead of January. Please bring the completion date forward by six weeks.


In [4]:
triage "$FROM" "$SUBJECT" "$BODY"


1. Project and project manager:


   - Project: P-100 — Office Building Hamburg


   - Customer: Nordbau Real Estate Ltd


   - Project manager: Anna Becker (anna.becker@example.com) 


2. Verdict: Critical.


   - According to the ontology’s business rules, a request to change the completion date is critical.


... (4 outputs left out)

   1. Escalate to Anna Becker within 1 working day of receipt of the email.


   2. Treat the request as a change request and route it for schedule/contract impact review.


   3. Reply to the client only after internal review; do not promise the earlier move-in date yet.


   4. If you need a precise calendar deadline, it depends on the email’s receipt timestamp; the ontology supports only the 1 working day requirement here. 


What did the agent do? The agent only uses the Fabric IQ tool. The ontology holds the data, the documents and the rules.


In [5]:
echo "$RESPONSE" | jq -r '.output[] | select(.type == "mcp_call") | "\(.name): \(.arguments)"'


ask_ontology: {"request":"1. Identify the project and project manager associated with the client's request: 'Our tenant wants to move in on 2027-11-01 instead of January. Please bring the completion date forward by six weeks.' Email from client@nordbau.example, subject: Earlier move-in.\n2. According to business rules in the ontology, assess whether bringing forward the completion date by six weeks is critical, watch, or not critical.\n3. Identify the rule that applies, quote the clause of the specification, offer, or contract that the request touches (related to completion dates, milestones, tenant handover, etc).\n4. Provide next steps, including responsible persons and deadlines."}


## 4. Email 2: a routine request
For contrast, an email that should not be critical.

The sender `team@isar.example` is made up for this demo, but it is the same address as in email E-4 (holiday schedule, project P-200) in `data/emails.csv`. That is how the agent can link the new email to the project.

A site plan (German: Lageplan) is the drawing of the building lot that shows where the building stands, the access roads and the neighbouring plots. The customer asks for the current version as a PDF file. No clause in the contract depends on it, so it is a routine request.


In [6]:
FROM="team@isar.example"
SUBJECT="Site plan"
BODY="Could you send us the current site plan as a PDF?"
echo "From:    $FROM"
echo "Subject: $SUBJECT"
echo
echo "$BODY"


From:    team@isar.example


Subject: Site plan


Could you send us the current site plan as a PDF?


In [7]:
triage "$FROM" "$SUBJECT" "$BODY"


1. Project and project manager.


- The only grounded project for the sender team@isar.example is: P-200 — Residential Complex Munich.


- Project manager: Ben Fischer.


2. Verdict: indeterminate


- The request for the current site plan as a PDF cannot be rated because there is no matching email or document in the database. If you meant the only grounded email (“Holiday schedule”), this is not critical.


3. Why:


... (4 outputs left out)

   1. Verify the exact email subject, date, and sender for the site-plan request. Responsible: Project Admin. Deadline: ASAP.


   2. If a matching email exists, route it to Ben Fischer for P-200. Responsible: Project Admin. Deadline: Unspecified (pending evidence).


   3. If you intended the grounded “Holiday schedule” email, no escalation is needed. Responsible: Project Admin. Deadline: None.


Cited according to Fabric IQ ontology and business rules: EmailCriticality .


## 4b. Email 3: an extra wish that should be a change request
This is email E-3 from `data/emails.csv`. The customer asks for something that the specification does not contain and claims that the fixed price covers it. The rule `ChangeRequestPolicy` in the ontology says what has to happen: charge it separately and do not absorb it into the fixed price. The agent prompt does not mention change requests.


In [1]:
FROM="client@nordbau.example"
SUBJECT="Additional request"
BODY="Please add a green roof with photovoltaics. This should be covered by the fixed price."
echo "From:    $FROM"
echo "Subject: $SUBJECT"
echo
echo "$BODY"


From:    client@nordbau.example
Subject: Additional request

Please add a green roof with photovoltaics. This should be covered by the fixed price.


In [9]:
ANSWER=$(triage "$FROM" "$SUBJECT" "$BODY")
echo "$ANSWER"


1. Project and project manager:


   - Project: P-100 — Office Building Hamburg


   - Project manager: Anna Becker (anna.becker@example.com)


2. Verdict: critical


3. Why: The request (“add a green roof with photovoltaics. This should be covered by the fixed price”) asks for new scope and contradicts the contract’s fixed-price clause.


   - Rule: EmailCriticality (change requests affecting scope and price are critical)


... (9 outputs left out)

      Deadline: within 1 working day


   4. Project team — if the client wants to proceed, prepare a change order/separate quote; assess impact on budget, design, and schedule before implementation.  


      Deadline: before any work starts


Citations: EmailCriticality rule, ChangeRequestPolicy rule, Offer clause (“Fixed price: EUR 2,400,000 net. … Changes outside the specification are charged by effort at EUR 85 per hour.”) .


The check below looks for the three things that only come from our configuration. The rule wording comes from `ChangeRequestPolicy`, the price comes from the offer document and the project manager comes from the tables. A language model does not repeat itself word for word, so a missing line is a hint to read the answer, not a proof of failure.


In [10]:
check() { echo "$ANSWER" | grep -qiE "$2" && echo "found    $1" || echo "MISSING  $1"; }
check "change request, charged separately (rule ChangeRequestPolicy)" "change request"
check "not absorbed into the fixed price (rule ChangeRequestPolicy)" "(not|never|n.t) (be )?(absorb|covered|included)|separate"
check "EUR 85 per hour (clause in the offer document)" "85"
check "project manager Anna Becker (table project_managers)" "Anna Becker"


found    change request, charged separately (rule ChangeRequestPolicy)


found    not absorbed into the fixed price (rule ChangeRequestPolicy)


found    EUR 85 per hour (clause in the offer document)


found    project manager Anna Becker (table project_managers)


### 4b.2 Live test: change the rule and watch the answer change
Matching words could also come from general knowledge. To prove that the answer follows the rule, the next cell changes the rule for a moment:

1. It uploads the ontology with a different statement for `ChangeRequestPolicy`: "It must be quoted with a 15 percent surcharge and approved in writing by the client before any work starts". Nothing else changes, same agent, same prompt, same data. The files in `ontology/` stay as they are, only the upload is modified.
2. It asks the agent the same question again.
3. It restores the rule from the files. This happens in an exit handler, so the rule is also restored when a step fails.

The cell takes a few minutes. If you stop it halfway, run `.venv/bin/python scripts/update_ontology.py` from the project folder to restore the rule.


In [11]:
(
  trap '../.venv/bin/python ../scripts/update_ontology.py && echo "Rule restored from ontology/rules/ChangeRequestPolicy.tmdl"' EXIT
  ../.venv/bin/python ../scripts/update_ontology.py --rule ChangeRequestPolicy \
    --statement "A customer request outside the specification is a change request. It must be quoted with a 15 percent surcharge and approved in writing by the client before any work starts." \
  || exit 1
  ANSWER=$(triage "$FROM" "$SUBJECT" "$BODY")
  echo "== Answer with the changed rule (lines that name the new requirements)"
  echo "$ANSWER" | grep -iE "15 ?(percent|%)|surcharge|in writing|written" || echo "(none found)"
)


Ontology updated (rule ChangeRequestPolicy overridden)


== Answer with the changed rule (lines that name the new requirements)


- Rule: ChangeRequestPolicy – Requests outside the project specification are treated as change requests and must be quoted with a 15% surcharge; written client approval is required before any work starts.


3. Anna Becker / Commercial Owner — Prepare a written change quote as per the 15% surcharge rule or using effort-based pricing from the offer.


4. Client (client@nordbau.example) — Provide written approval before any work starts.


Ontology updated (files unchanged)


Rule restored from ontology/rules/ChangeRequestPolicy.tmdl


## 5. How the answer comes about

Two systems work together. Foundry hosts the model and the agent. Fabric holds the ontology and the data. The cells below show live evidence for every step, taken from the last call of this notebook (the site plan email).

| Step | Where | Which of our configurations is used | Evidence |
|------|-------|-------------------------------------|----------|
| 1. The notebook sends the email to the agent by name (`agent_reference`) | Foundry | The current version of agent `project-planning-agent`: model `gpt-4.1`, the prompt from `scripts/agent_instructions.txt` and the tool `fabric_iq_preview` | 5.1 |
| 2. The model reads the prompt and the tool list, then writes a question for the tool | Foundry | The prompt tells it to answer through the Fabric IQ tool. The tool list (`ask_ontology`, `list_ontology_rules`, `list_ontology_entities`) comes from the Fabric endpoint. Only the question the model writes reaches Fabric, not the prompt and not the original email text. | 5.2, 5.3 |
| 3. Foundry calls the ontology endpoint with your Entra token | Foundry to Fabric | Connection `fabric-iq-ontology` (`UserEntraToken`, target = ontology endpoint), created by `scripts/setup.py` | 5.5 (login name) |
| 4. Fabric reads the ontology and plans the answer | Fabric | The `ontology/` folder: descriptions and synonyms (`entities/`), relationships, business rules (`rules/`) and data bindings (`tables/`) | 5.4, 5.6 |
| 5. Fabric runs T-SQL on the lakehouse as you | Fabric | The bindings map entities to tables. The data comes from `data/`, loaded by `scripts/setup.py`. | 5.5 |
| 6. Fabric answers with the rules applied, the model writes the final text | Fabric, Foundry | Rules such as `EscalationPolicy` show up in the Fabric answer. The prompt rules (cite sources, do not invent data) shape the final text. | 5.2, 5.6 |


### 5.1 Step 1: the agent configuration that Foundry applied
The Responses API echoes the configuration of the agent version that answered. Compare it with `scripts/agent_instructions.txt` and `infra/main.bicep` (model deployment).


In [12]:
echo "$RESPONSE" | jq -r '"Agent:  \(.agent_reference.name), version \(.agent_reference.version)\nModel:  \(.model)\nTool:   \(.tools[0].type), connection \(.tools[0].project_connection_id | split("/") | last)\n\nPrompt:\n\(.instructions)"'


Agent:  project-planning-agent, version 3


Model:  gpt-4.1


Tool:   fabric_iq_preview, connection fabric-iq-ontology


Prompt:


You are a project planning assistant for construction projects.


Answer every question with the Fabric IQ tool (the project ontology). Cite the documents or clauses you rely on.


When you call the tool, write a complete question and include the details of the user's message word for word, such as sender addresses, subjects, names and dates. Do not summarize or leave out details, because the tool needs them to find the project.


Do not invent data. If the ontology does not contain the information, say so.


### 5.2 Steps 2 and 6: what the model sent to Fabric and what came back
The model writes the tool request itself. Fabric never sees the prompt or the original email, only this text. If the model leaves out a detail such as the sender address, Fabric cannot use it. The agent prompt therefore tells the model to include the email details word for word. Compare the request with the email above.


In [13]:
echo "$RESPONSE" | jq -r '.output[] | select(.type == "mcp_call") | "Tool: \(.name)\n\nRequest written by the model:\n\(.arguments | fromjson | .request)\n\nAnswer from Fabric:\n\(.output | fromjson | .response)"'


Tool: ask_ontology


Request written by the model:


Identify the project and project manager for the request from team@isar.example regarding 'Site plan', dated today, asking for the current site plan as a PDF. Determine if this is critical, watch or not critical according to the project ontology business rules. Name the rule that applies and quote the relevant clause in the specification, offer, or contract. List next steps including responsible persons and deadlines.


Answer from Fabric:


## Grounded match


I couldn’t find a **today-dated** email from **team@isar.example** about **“Site plan”** or a **current site plan PDF**.


... (15 outputs left out)

- **Verify the exact email** subject/date/sender for the site-plan request.


- If a matching email exists, **route it to Ben Fischer** for P-200.


- **Deadline:** not grounded from the available evidence.


- If you intended the grounded **holiday schedule** email, no escalation is needed.


### 5.3 The trace of the call in Application Insights
Foundry writes a server-side trace for every agent call, because `infra/main.bicep` connects Application Insights to the project (connection `appinsights`). The trace has one span per stage. The cell waits until the trace is ingested, which takes a few minutes. It finds the spans through the response ID.


In [14]:
WORKSPACE_ID=$(output logAnalyticsWorkspaceId)
RESPONSE_ID=$(echo "$RESPONSE" | jq -r .id)

query_logs() {
  az rest --method post --resource https://api.loganalytics.io \
    --url "https://api.loganalytics.io/v1/workspaces/$WORKSPACE_ID/query" \
    --headers "Content-Type=application/json" \
    --body "$(jq -n --arg q "$1" '{query: $q}')"
}

get_spans() {
  SPANS=$(query_logs "AppDependencies | where Properties['gen_ai.response.id'] == '$RESPONSE_ID' | order by TimeGenerated asc | project TimeGenerated, Name, DurationMs, Properties")
}

# The invoke_agent span is written last, so wait for it, then let the other spans arrive
for i in $(seq 30); do
  get_spans
  [ "$(echo "$SPANS" | jq '[.tables[0].rows[] | select(.[1] | startswith("invoke_agent"))] | length')" -gt 0 ] && break
  echo "Trace not ingested yet, waiting 20 s ..."; sleep 20
done
sleep 15; get_spans

echo "Response ID: $RESPONSE_ID"
echo "$SPANS" | jq -r '.tables[0].rows[] | [.[0], .[1], "\(.[2] / 1000 | round) s"] | @tsv' | column -t -s $'\t'


Response ID: resp_01554c666a3cf867006ae9d0bf676c819097761d91fd128fcd


2026-11-03T12:17:03.7866993Z  invoke_agent project-planning-agent:3             76 s


2026-11-03T12:17:03.7870327Z  execute_tool mcp_fabric-iq-ontology.ask_ontology  73 s


2026-11-03T12:18:17.0042913Z  chat gpt-4.1-2025-04-14                           2 s


The `invoke_agent` span covers the whole call. Inside it, `execute_tool` is the call to Fabric and `chat` is the model writing the final answer. Almost all of the time sits in the Fabric call. The details of each span:


In [15]:
echo "$SPANS" | jq -r '.tables[0].rows[] | .[1] as $span | (.[3] | fromjson) as $a
  | "[\($span)]",
    (if $a["gen_ai.tool.call.arguments"] then "  tool arguments: \($a["gen_ai.tool.call.arguments"])" else empty end),
    (if $a["gen_ai.usage.input_tokens"] then "  tokens in/out:  \($a["gen_ai.usage.input_tokens"]) / \($a["gen_ai.usage.output_tokens"])" else empty end),
    (if $span | startswith("chat") then ($a["gen_ai.input.messages"] | fromjson | .[] | "  input \(.role): \(.parts[0] | (.content // .name // .type) | tostring | .[0:90] | gsub("\n"; " "))") else empty end)'


[invoke_agent project-planning-agent:3]


[execute_tool mcp_fabric-iq-ontology.ask_ontology]


  tool arguments: {"request":"Identify the project and project manager for the request from team@isar.example regarding 'Site plan', dated today, asking for the current site plan as a PDF. Determine if this is critical, watch or not critical according to the project ontology business rules. Name the rule that applies and quote the relevant clause in the specification, offer, or contract. List next steps including responsible persons and deadlines."}


[chat gpt-4.1-2025-04-14]


  tokens in/out:  1903 / 315


  input system: You are a project planning assistant for construction projects.  Answer every question wit


  input user: We received the email below. Answer in this structure: 1. Project and project manager. 2. 


  input assistant: mcp_fabric-iq-ontology.ask_ontology


  input tool: tool_call_response


### 5.4 Step 4: the configuration inside Fabric, as Fabric returns it
The ontology endpoint is an MCP server. The cell calls it directly, without Foundry, with your own token. `list_ontology_rules` and `list_ontology_entities` return what `scripts/setup.py` uploaded from the `ontology/` folder: the rules, the descriptions, the synonyms and the bindings.


In [16]:
FABRIC_TOKEN=$(az account get-access-token --resource https://api.fabric.microsoft.com --query accessToken -o tsv)
fabric() { curl -s -H "Authorization: Bearer $FABRIC_TOKEN" "https://api.fabric.microsoft.com/v1$1"; }
WS=$(fabric /workspaces | jq -r '.value[] | select(.displayName == "ProjectPlanning") | .id')
ONT=$(fabric /workspaces/$WS/ontologies | jq -r '.value[] | select(.displayName == "ProjectPlanning") | .id')

mcp() {
  curl -s "https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/$WS/items/$ONT/ontologyEndpoint" \
    -H "Authorization: Bearer $FABRIC_TOKEN" -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream" \
    -d "$(jq -n --arg tool "$1" --argjson args "$2" '{jsonrpc: "2.0", id: 1, method: "tools/call", params: {name: $tool, arguments: $args}}')"
}

echo "== Rules (ontology/rules/*.tmdl)"
mcp list_ontology_rules '{}' | jq -r '.result.structuredContent.values[] | "\(.name): \(.statement)"'
echo
echo "== Entity Email (ontology/entities/Email.tmdl)"
mcp list_ontology_entities '{"entityName": "Email"}' | jq -r '.result.structuredContent.values[0] | "description:   \(.description)\nsynonyms:      \(.synonyms | join(", "))\nbacking table: \(.backingTable)\nkey:           \(.keyProperties | join(", "))\nproperties:    \(.properties | map(.name + " (" + .description + ")") | join("; "))"'


== Rules (ontology/rules/*.tmdl)


EmailCriticality: An email is critical if it affects the completion date, the budget or the scope of its project, or if it contradicts an assumption or clause in the specification, offer or contract of that project. An email that only needs to be monitored is rated watch. All other emails are rated not critical.


EscalationPolicy: Every critical email must be escalated to the project manager of its project within 1 working day.


ChangeRequestPolicy: A customer request outside the specification is a change request. It must be charged separately and must not be absorbed into the fixed price.


== Entity Email (ontology/entities/Email.tmdl)


description:   An incoming email about a project.


synonyms:      Mail, Message


backing table: emails


key:           MailId


properties:    Body (Text of the email.); MailId (Unique email ID.); ProjectId (ID of the project the email refers to.); ReceivedAt (Date the email was received (YYYY-MM-DD).); Sender (Email address of the sender.); Subject (Subject line.)


### 5.5 Step 5: the Fabric log, the SQL behind this call
I found no separate log for ontology calls on this path. What Fabric records is every T-SQL statement that reached the lakehouse SQL endpoint, in `queryinsights.exec_requests_history`. The ontology is bound to the semantic model, so it turns the model's question into DAX, and the semantic model reads the rows from the lakehouse with such statements. This is the closest view into Fabric. The cell shows the statements in the time window of the `invoke_agent` span. Look for three things:

* `login_name` is you, because the connection passes your token through.
* `dbo.projects`, `dbo.emails`, `dbo.documents` and the joins come from the tables and relationships of the semantic model. Some statements end with a label that names the `DatasetId` of the semantic model and the operation `DatasetExecuteQueries`.
* The statements take milliseconds. The rest of the time in the Fabric call is not SQL.

The cell needs the command line tool `sqlcmd` (go-sqlcmd). It downloads the Linux x64 build to `.tools/`. On other systems install go-sqlcmd and point `SQLCMD` to it.


In [17]:
SQLCMD=../.tools/sqlcmd
if [ ! -x "$SQLCMD" ]; then
  mkdir -p ../.tools
  curl -sL https://github.com/microsoft/go-sqlcmd/releases/download/v1.10.0/sqlcmd-linux-amd64.tar.bz2 | tar xj -C ../.tools sqlcmd
fi

LH=$(fabric /workspaces/$WS/lakehouses | jq -r '.value[] | select(.displayName == "projectdata") | .id')
SQL_SERVER=$(fabric /workspaces/$WS/lakehouses/$LH | jq -r '.properties.sqlEndpointProperties.connectionString')
START=$(echo "$SPANS" | jq -r '.tables[0].rows[] | select(.[1] | startswith("invoke_agent")) | .[0]')
SECONDS_TOTAL=$(echo "$SPANS" | jq -r '.tables[0].rows[] | select(.[1] | startswith("invoke_agent")) | .[2] / 1000 | ceil')

# Query Insights can lag behind by a short time
for i in $(seq 15); do
  SQL_LOG=$("$SQLCMD" -S "$SQL_SERVER" -d projectdata --authentication-method ActiveDirectoryAzCli -W -s " | " -Q "
    SET NOCOUNT ON;
    DECLARE @t0 datetime2 = CONVERT(datetime2, '${START%Z}', 126);
    SELECT CONVERT(varchar(12), start_time, 114) AS start_utc, login_name, row_count AS [rows], total_elapsed_time_ms AS ms,
           LEFT(REPLACE(REPLACE(SUBSTRING(command, CHARINDEX('FROM', command), 400), CHAR(10), ' '), '''''', ''''), 200) AS tables_and_filter
    FROM queryinsights.exec_requests_history
    WHERE start_time BETWEEN @t0 AND DATEADD(second, $SECONDS_TOTAL, @t0) AND command NOT LIKE 'EXEC sp_set_session_context%'
    ORDER BY start_time" | grep -v "^Statement ID")
  [ "$(echo "$SQL_LOG" | wc -l)" -gt 3 ] && break
  echo "No statements visible yet, waiting 20 s ..."; sleep 20
done

echo "Window: $START + $SECONDS_TOTAL s (UTC)"
echo "$SQL_LOG"


Window: 2026-11-03T12:17:03.7866993Z + 76 s (UTC)


start_utc login_name rows ms tables_and_filter


--------- ---------- ---- -- -----------------


12:17:30.920 ga1@cptazure.org 0 158 FROM dbo.emails e     INNER JOIN dbo.projects p         ON p.ProjectId = e.ProjectId     INNER JOIN dbo.project_managers pm         ON pm.ManagerId = p.ManagerId     WHERE e.Sender = 'team@isar.exampl


12:17:32.650 ga1@cptazure.org 1 53 FROM dbo.emails e WHERE e.ReceivedAt = '2026-11-03' ORDER BY e.MailId


12:17:35.529 ga1@cptazure.org 5 126 FROM dbo.emails e ORDER BY e.ReceivedAt DESC, e.MailId


12:17:41.392 ga1@cptazure.org 0 99 FROM dbo.documents d WHERE LOWER(d.Title) LIKE '%site plan%'    OR LOWER(d.Content) LIKE '%site plan%' ORDER BY d.ProjectId, d.DocId


12:17:46.612 ga1@cptazure.org 2 115 FROM dbo.emails e WHERE LOWER(e.Sender) LIKE '%isar%'    OR LOWER(e.Subject) LIKE '%site%'    OR LOWER(e.Body) LIKE '%site%' ORDER BY e.ReceivedAt DESC, e.MailId


12:17:50.177 ga1@cptazure.org 4 119 FROM dbo.documents d INNER JOIN dbo.projects p     ON p.ProjectId = d.ProjectId ORDER BY d.ProjectId, d.DocId


12:17:56.745 ga1@cptazure.org 2 107 FROM dbo.projects p LEFT JOIN dbo.project_managers pm     ON pm.ManagerId = p.ManagerId ORDER BY p.ProjectId


12:18:02.552 ga1@cptazure.org 0 106 FROM dbo.documents d INNER JOIN dbo.projects p     ON p.ProjectId = d.ProjectId WHERE LOWER(d.Content) LIKE '%plan%'    OR LOWER(d.Title) LIKE '%plan%' ORDER BY d.ProjectId, d.DocType


#### How to read this log
Each row is one T-SQL statement that Fabric ran on the lakehouse while it prepared the answer. Read the rows from top to bottom, they follow the steps of the question.

| Column | Meaning |
|--------|---------|
| `start_utc` | When the statement started. The first one starts well after the call began, the time before is not SQL. |
| `login_name` | Who ran the statement. This is your user, because Foundry passes your token through. |
| `rows` | Number of rows returned. `0` means that nothing matched, for example a keyword that appears in no document. |
| `ms` | Run time of the statement. |
| `tables_and_filter` | The `FROM` and `WHERE` part: which of our tables Fabric read and how it filtered them. |

What the log proves:

* Fabric reads only the four tables of the ontology (`dbo.emails`, `dbo.projects`, `dbo.documents`, `dbo.project_managers`). The answer is built from our data, not from model memory.
* A typical sequence is: look up the sender in `emails` to find the project, search `documents` for a keyword, then read all documents of that project (`WHERE d.ProjectId = 'P-200'`) and its project manager. A keyword search with `0` rows, for example `site plan`, tells you that no document mentions it.
* The statements run as you. A user without access to the workspace would get no rows.

What the log does not show: how Fabric applied the business rules. The rules show up in the answer text (5.2) and in the live test (5.6).


### 5.6 Live test: change a rule and watch the answer change
To prove that the answer uses our configuration and not general knowledge, I changed one rule in `ontology/rules/EscalationPolicy.tmdl` from "within 1 working day" to "by phone within 2 hours", uploaded the ontology with `updateDefinition` and asked the agent again. Nothing else changed: same agent, same prompt, same data.

The agent answered: "A critical email must be escalated by phone to the project manager within 2 hours." The text "by phone" appears nowhere in the prompt or in the data. It came from the rule in the ontology. I then restored the original rule and uploaded the ontology again.

The same run showed a limit. The model wrote its tool request without the sender address, so Fabric could not match the email to a project and said that no project "Nordbau" exists. The agent prompt now tells the model to write a complete question and to include the email details word for word. In five test runs with that sentence the sender reached Fabric every time. A first version of the sentence, "pass the message unchanged", failed: the model then sent only the email text without a question, and Fabric answered "I can help only with Azure-related topics".


## 6. A direct GQL request to the graph

Section 5 showed that the agent does not use the graph: Fabric answers its questions with SQL. To see what the graph can do, this section sends a query to it directly, without the agent. The graph is the copy of an ontology that you materialize under **Manage graph** in the portal. Each entity type is a node type and each relationship is an edge type. The graph needs an ontology that is bound to the lakehouse. The demo ontology is bound to the semantic model, so the cells below work only while a graph from the earlier experiment exists in the workspace (see "Graph experiment" in `docs/how-it-works.md`). Without a graph the cell says so, and a graph that was never materialized fails with `GraphNotQueryable`.

The request goes to the GQL query API of Fabric: a POST to `/workspaces/{workspace}/GraphModels/{graph}/executeQuery` with the body `{"query": "..."}`. GQL is the international standard for graph queries (ISO/IEC 39075). A pattern like `(a:Task)-[:Task_Depends_On]->(b:Task)` reads as "a task a that depends on a task b".

* The first query follows `Task_Depends_On` for one to ten steps and returns every task that depends on T-101, directly or indirectly. This is the question that took a join in SQL. In GQL the number of steps is part of the pattern: `->{1,10}`.
* The second query crosses two edge types, from a task to its project and on to the project manager. `Project` is a reserved word in GQL, so the label needs backticks.
* The third query shows the difference between direct and indirect: only the tasks that follow T-103 in one step.


In [ ]:
FABRIC_TOKEN=$(az account get-access-token --resource https://api.fabric.microsoft.com --query accessToken -o tsv)
fabric() { curl -s -H "Authorization: Bearer $FABRIC_TOKEN" "https://api.fabric.microsoft.com/v1$1"; }
WS=$(fabric /workspaces | jq -r '.value[] | select(.displayName == "ProjectPlanning") | .id')
GRAPH=$(fabric /workspaces/$WS/items | jq -r '.value[] | select(.type == "GraphModel") | .id')
[ -n "$GRAPH" ] || echo "No graph model in the workspace. See the section Graph experiment in docs/how-it-works.md."

# Sends one GQL query and prints the result table, or the error
gql() {
  curl -s -X POST "https://api.fabric.microsoft.com/v1/workspaces/$WS/GraphModels/$GRAPH/executeQuery?preview=true" \
    -H "Authorization: Bearer $FABRIC_TOKEN" -H "Content-Type: application/json" \
    -d "$(jq -n --arg q "$1" '{query: $q}')" |
    jq -r 'if .result then (.result.columns | map(.name)) as $c | ($c | join(" | ")), (.result.data[] | [.[$c[]]] | map(tostring) | join(" | ")) else (.errorCode // .status.cause.description) end'
}

echo "== Tasks that depend directly or indirectly on T-101"
gql "MATCH (t:Task)-[:Task_Depends_On]->{1,10}(s:Task {TaskId: 'T-101'})
     RETURN DISTINCT t.TaskId AS TaskId, t.Name AS Name, t.CostEur AS CostEur ORDER BY TaskId"
echo
echo "== Project and project manager of task T-104 (two edges)"
gql "MATCH (t:Task {TaskId: 'T-104'})-[:Task_Of_Project]->(p:\`Project\`)-[:Project_Managed_By]->(m:ProjectManager)
     RETURN t.Name AS Task, p.Name AS ProjectName, m.Name AS Manager"
echo
echo "== Tasks that follow T-103 in one step"
gql "MATCH (a:Task)-[:Task_Depends_On]->(b:Task {TaskId: 'T-103'}) RETURN a.TaskId AS DirectSuccessor ORDER BY DirectSuccessor"


## Summary
* The agent prompt contains no rating scheme. The rules `EmailCriticality`, `EscalationPolicy` and `ChangeRequestPolicy` live in the ontology and are applied on the Fabric side.
* Section 4b shows this for `ChangeRequestPolicy`. The agent treats the green roof wish as a change request, and when the rule text is replaced for a moment, the answer repeats the new requirements. Section 5.6 does the same for `EscalationPolicy`.
* The project, its manager and the documents come from the ontology tables, so the answer cites the clauses of the specification, offer and contract.
* Foundry runs the model and the prompt. Fabric runs the ontology and the SQL. The trace shows that most of the time is spent in the Fabric call.
* The materialized graph answers path questions with GQL (section 6), but the agent does not use it: with the graph in place Fabric still ran SQL for the same question.
